Data Preparation

In [9]:
! pip install pandas
! pip install torch
! pip install transformers



Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 25.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 435.5/435.5 KB 44.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 KB 31.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 KB 40.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.5/776.5 KB 42.0 MB/s eta 0:00:00


In [7]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/home/dion/projects/transpoly/TransPolymer/data/Eea.csv')
properties = df.drop(columns=['smiles'])
smiles_strings = df['smiles']

# Example: Convert to tensors if using PyTorch
import torch
from torch.utils.data import Dataset, DataLoader

class PolymerDataset(Dataset):
    def __init__(self, properties, smiles):
        self.properties = properties
        self.smiles = smiles
    
    def __len__(self):
        return len(self.smiles)
    
    def __getitem__(self, idx):
        return self.properties.iloc[idx].values, self.smiles[idx]

dataset = PolymerDataset(properties, smiles_strings)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


Model Fine-Tuning

In [10]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader

# Define the number of epochs and maximum length
num_epochs = 10  # Adjust this number based on your training needs
max_length = 128  # Adjust this based on your dataset

# Load pre-trained GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Add padding token to the tokenizer
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Example: Convert to tensors if using PyTorch
class PolymerDataset(Dataset):
    def __init__(self, properties, smiles):
        self.properties = properties
        self.smiles = smiles
    
    def __len__(self):
        return len(self.smiles)
    
    def __getitem__(self, idx):
        return self.properties.iloc[idx].values, self.smiles[idx]

dataset = PolymerDataset(properties, smiles_strings)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Extend tokenizer to handle property input (optional, for custom tokens)
additional_tokens = [f"<prop_{i}>" for i in range(properties.shape[1])]
tokenizer.add_tokens(additional_tokens)
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.resize_token_embeddings(len(tokenizer))

# Training setup
optimizer = AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

def encode_properties(properties, tokenizer):
    """Encode properties as token IDs."""
    prop_tokens = [f"<prop_{i}>" for i, prop in enumerate(properties)]
    prop_ids = tokenizer.convert_tokens_to_ids(prop_tokens)
    return prop_ids

# Fine-tuning loop
model.train()
for epoch in range(num_epochs):
    for props, smiles in dataloader:
        prop_ids = [encode_properties(p, tokenizer) for p in props]
        
        # Tokenize SMILES strings
        inputs = tokenizer(smiles, return_tensors='pt', padding=True, truncation=True, max_length=max_length)
        
        # Concatenate property tokens with SMILES tokens
        input_ids = []
        attention_masks = []
        for i in range(len(inputs['input_ids'])):
            combined_input = prop_ids[i] + inputs['input_ids'][i].tolist()
            # Pad the combined input to max_length
            combined_input = combined_input[:max_length] + [tokenizer.pad_token_id] * (max_length - len(combined_input))
            input_ids.append(combined_input)
            # Create an attention mask
            attention_mask = [1] * len(prop_ids[i]) + inputs['attention_mask'][i].tolist()
            attention_mask = attention_mask[:max_length] + [0] * (max_length - len(attention_mask))
            attention_masks.append(attention_mask)

        input_ids = torch.tensor(input_ids)
        attention_masks = torch.tensor(attention_masks)

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_masks, labels=input_ids)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

# Save the fine-tuned model
model.save_pretrained('polymer_gpt')



/home/dion/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/dion/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/home/dion/.local/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1, Loss: 0.957649827003479
Epoch 2, Loss: 0.6419196724891663
Epoch 3, Loss: 0.5198633074760437
Epoch 4, Loss: 0.38764670491218567
Epoch 5, Loss: 0.2976529002189636
Epoch 6, Loss: 0.24233466386795044
Epoch 7, Loss: 0.20079192519187927
Epoch 8, Loss: 0.17633797228336334
Epoch 9, Loss: 0.15773345530033112
Epoch 10, Loss: 0.1382562667131424


Generating SMILES Strings

In [11]:
model.eval()

def generate_smiles(properties, tokenizer, model, max_length=100):
    prop_ids = encode_properties(properties, tokenizer)
    input_ids = torch.tensor([prop_ids])
    
    # Generate SMILES string
    outputs = model.generate(input_ids, max_length=max_length, num_return_sequences=1)
    smiles = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return smiles

# Example usage
example_properties = [0.5]  # Replace with actual property values
generated_smiles = generate_smiles(example_properties, tokenizer, model)
print(f"Generated SMILES: {generated_smiles}")


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated SMILES: <prop_0> *Cc1ccc(C(*)=S)cc1
